In [ ]:
file_path = '/content/sample_data/vendor_policy.txt'
with open(file_path, 'r') as f:
    text_content = f.read()

print(f"Successfully loaded text from {file_path}. Total characters: {len(text_content)}")

In [ ]:
def fixed_size_chunking(text, chunk_size, overlap):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
        if start >= len(text):
            break
    return chunks

chunk_size = 200
overlap = 50
chunks = fixed_size_chunking(text_content, chunk_size, overlap)

print(f"Total Fixed-size Chunks: {len(chunks)}\n")

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} (length: {len(chunk)}) ---")
    print(chunk)
    print("\n")

In [ ]:
# Install NLTK if not already installed
!pip install nltk

In [ ]:
# Download the 'punkt' tokenizer for sentence tokenization
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
from nltk.tokenize import sent_tokenize

sentences = sent_tokenize(text_content)

print(f"Total Sentence-based Chunks: {len(sentences)}\n")

print("--- Sentence-based Chunks ---")
for i, sentence in enumerate(sentences):
    print(f"Sentence {i+1} (length: {len(sentence)}):")
    print(sentence)
    print("\n")

In [ ]:
import re

def section_based_chunking(text):
    # Regex to find sections starting with '1. ', '2. ', '3. ', '4. ' etc.
    # It captures the section header and the content until the next header or end of text.
    sections = re.split(r'\n\n(\d+\.\s[A-Za-z]+\s[A-Za-z]+.*)', text)

    # The first element will be text before the first header, or empty if text starts with a header
    # We need to pair headers with their content.
    # Let's clean up and pair them
    chunks = []
    current_header = None
    for part in sections:
        if part.strip() == '':
            continue
        if re.match(r'^\d+\.\s[A-Za-z]+\s[A-Za-z]+.*', part):
            current_header = part.strip()
        else:
            if current_header:
                chunks.append(f"{current_header}\n{part.strip()}")
                current_header = None # Reset after appending
            else:
                # Handle any leading text that is not part of a numbered section if it exists
                chunks.append(part.strip())

    # Handle case where the text starts with a section header without any preceding text
    if not chunks and sections[0].strip() == '' and len(sections) > 1:
        # Re-process to correctly associate headers with their content when re.split creates an empty first element
        # This approach is a bit more robust:

        # Find all section headers and their starting positions
        matches = list(re.finditer(r'\n\n(\d+\.\s[A-Za-z]+\s[A-Za-z]+.*)', text))

        chunks = []
        start_idx = 0
        for i, match in enumerate(matches):
            # Content from the start_idx up to the current match's start
            if i == 0 and match.start() > 0:
                chunks.append(text[start_idx:match.start()].strip())

            # The section header and its content until the next header or end of text
            section_start = match.start(1) # Start of the captured group (the header itself)
            section_end = matches[i+1].start() if i + 1 < len(matches) else len(text)
            chunks.append(text[section_start:section_end].strip())




    return [chunk for chunk in chunks if chunk]

semantic_chunks = section_based_chunking(text_content)

print(f"Total Semantic/Section-based Chunks: {len(semantic_chunks)}\n")

print("--- Semantic/Section-based Chunks ---")
for i, chunk in enumerate(semantic_chunks):
    print(f"--- Section {i+1} (length: {len(chunk)}) ---")
    print(chunk)
    print("\n")

In [ ]:
# Install the sentence-transformers library
!pip install sentence-transformers

In [ ]:
from huggingface_hub import login
login(token="YOUR_API_KEY")

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the 'all-MiniLM-L6-v2' model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model 'all-MiniLM-L6-v2' loaded successfully.")

### Generating Embeddings for Fixed-size Chunks

In [ ]:
# Generate embeddings for fixed-size chunks
fixed_size_chunk_embeddings = embedding_model.encode(chunks)

print(f"Fixed-size chunk embeddings shape: {fixed_size_chunk_embeddings.shape}")
print(f"Sample embedding for first fixed-size chunk:\n{fixed_size_chunk_embeddings[0][:5]}...") # Display first 5 dimensions of the first embedding

### Generating Embeddings for Sentence-based Chunks

In [ ]:
# Generate embeddings for sentence-based chunks
sentence_chunk_embeddings = embedding_model.encode(sentences)

print(f"Sentence-based chunk embeddings shape: {sentence_chunk_embeddings.shape}")
print(f"Sample embedding for first sentence-based chunk:\n{sentence_chunk_embeddings[0][:5]}...") # Display first 5 dimensions of the first embedding

### Generating Embeddings for Semantic/Section-based Chunks

In [ ]:
# Generate embeddings for semantic/section-based chunks
semantic_chunk_embeddings = embedding_model.encode(semantic_chunks)

print(f"Semantic/Section-based chunk embeddings shape: {semantic_chunk_embeddings.shape}")
print(f"Sample embedding for first semantic/section-based chunk:\n{semantic_chunk_embeddings[0][:5]}...") # Display first 5 dimensions of the first embedding

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

query = "What happens if my invoice doesn't have a PO number?"
query_embedding = embedding_model.encode([query])[0]

### Cosine Similarity for Fixed-size Chunks

In [ ]:
fixed_size_similarities = cosine_similarity(query_embedding.reshape(1, -1), fixed_size_chunk_embeddings)
top_fixed_size_chunk_idx = np.argmax(fixed_size_similarities)
top_fixed_size_score = fixed_size_similarities[0, top_fixed_size_chunk_idx]
top_fixed_size_chunk = chunks[top_fixed_size_chunk_idx]

print(f"--- Top Fixed-size Chunk Match (Score: {top_fixed_size_score:.4f}) ---")
print(top_fixed_size_chunk)
print("\n")

### Cosine Similarity for Sentence-based Chunks

In [ ]:
sentence_similarities = cosine_similarity(query_embedding.reshape(1, -1), sentence_chunk_embeddings)
top_sentence_chunk_idx = np.argmax(sentence_similarities)
top_sentence_score = sentence_similarities[0, top_sentence_chunk_idx]
top_sentence_chunk = sentences[top_sentence_chunk_idx]

print(f"--- Top Sentence-based Chunk Match (Score: {top_sentence_score:.4f}) ---")
print(top_sentence_chunk)
print("\n")

### Cosine Similarity for Semantic/Section-based Chunks

In [ ]:
semantic_similarities = cosine_similarity(query_embedding.reshape(1, -1), semantic_chunk_embeddings)
top_semantic_chunk_idx = np.argmax(semantic_similarities)
top_semantic_score = semantic_similarities[0, top_semantic_chunk_idx]
top_semantic_chunk = semantic_chunks[top_semantic_chunk_idx]

print(f"--- Top Semantic/Section-based Chunk Match (Score: {top_semantic_score:.4f}) ---")
print(top_semantic_chunk)
print("\n")

## Summary of Chunking Strategies and Cosine Similarity Results

We explored three different text chunking strategies and evaluated their effectiveness for a specific query using cosine similarity.

**Query:** "What happens if my invoice doesn't have a PO number?"

### Results:

*   **Fixed-size Chunks:**
    *   **Top Match Score:** 0.6552
    *   **Top Chunk:** "t include a valid purchase order number, itemized charges, and supporting documentation. Invoices without a valid PO number will be rejected and returned to the vendor. 2. Payment Terms Standard paym"
    *   **Observations:** While it captured relevant information, it included a partial sentence and some irrelevant text from the next section due to its arbitrary size. This can lead to less precise results.

*   **Sentence-based Chunks:**
    *   **Top Match Score:** 0.8010
    *   **Top Chunk:** "Invoices without a valid PO number will be rejected and returned to the vendor."
    *   **Observations:** This strategy yielded the highest similarity score and provided the most concise and accurate answer to the query. It directly matched the core information needed.

*   **Semantic/Section-based Chunks:**
    *   **Top Match Score:** 0.5320
    *   **Top Chunk:** "1. Invoice Submission All invoices must be submitted through the vendor portal within 30 days of service delivery. Invoices must include a valid purchase order number, itemized charges, and supporting documentation. Invoices without a valid PO number will be rejected and returned to the vendor."
    *   **Observations:** Although this chunk is semantically relevant, it contains a lot of additional context. The larger size of the semantic chunk diluted its similarity score for this very specific query compared to a shorter, more precise sentence-based chunk.

### Tradeoffs and Recommendations:

1.  **Fixed-size Chunking (e.g., 200 characters with 50 overlap):**
    *   **Tradeoffs:** Simple to implement, guarantees uniform chunk sizes. However, it can cut sentences or topics mid-flow, leading to incomplete information or irrelevant fragments. Overlap helps mitigate this but doesn't solve the core issue of semantic coherence.
    *   **Recommendation:** Best for very large, unstructured documents where a quick and dirty chunking method is needed, and where the precise semantic boundary isn't critical. Also useful as a fallback when other methods fail or are too complex to implement.

2.  **Sentence-based Chunking:**
    *   **Tradeoffs:** Provides semantically coherent units (complete thoughts/statements), which often leads to higher precision in retrieval for specific questions. However, sentences can sometimes be too short or too long, potentially splitting related ideas or combining unrelated ones.
    *   **Recommendation:** Ideal for question-answering systems or retrieval-augmented generation (RAG) where precise, atomic units of information are desired. Works well when the density of information within sentences is high and queries are specific.

3.  **Semantic/Section-based Chunking:**
    *   **Tradeoffs:** Ensures that chunks maintain logical and hierarchical integrity, keeping related ideas together. This is great for understanding broader topics. However, these chunks can be quite large, potentially embedding too much irrelevant information for highly specific queries, thus lowering similarity scores.
    *   **Recommendation:** Excellent for building knowledge bases where context and completeness of topics are paramount. Suitable for queries that require a broader understanding of a section or for navigating documents structurally (e.g., "Tell me about Payment Terms"). It can also be combined with sub-chunking (e.g., sentence chunking within a semantic chunk) for more granular retrieval within a relevant section.